# RNN을 이용한 짧은 문자열 생성기
- week06에서 만든 RNN class를 그대로 재사용하고 출력층과 손실함수만 바꿔서 만들었습니다.
- 지금까지는 정답이 실수 하나인 회귀 문제였다면 이번에는 다음 글자가 뭘지 맞히는 분류 문제입니다.

## 언어모델과 문장 생성의 차이
- 언어모델(Language Model): 지금까지 나온 글자들을 보고 다음 글자의 확률분포를 계산하는 모델입니다. RNN의 순전파 + 학습 과정이 이 부분입니다.
- 문장 생성: 학습된 언어모델에서 확률분포를 뽑고, 그중 하나를 샘플링해서 다시 입력으로 넣는 과정을 반복하는 것입니다(자기회귀). 언어모델 자체는 아무것도 만들어내지 않고, 이 반복 절차가 있어야 문장이 생성됩니다.

## 기존에 배운 것 이외에 추가로 필요한 개념
1. 원-핫 벡터: 글자를 vocab 크기만큼의 벡터로, 그 글자 자리만 1로 표현합니다.
2. many-to-many 구조: 시퀀스 전체를 한 칸 밀어서 입력과 정답을 만듭니다 (입력 "hell" -> 정답 "ello") h의 다음 글자는 e 이므로 h가 입력이면 e가 정답인 구조입니다.
3. softmax: 출력 점수(z)를 확률 분포로 바꿔주는 함수입니다.
4. cross-entropy: 정답 글자의 예측 확률만 떼서 -log를 취하는 분류용 손실함수입니다.
5. softmax+cross-entropy의 역전파: 두 개를 같이 쓰면 dz = 예측확률 - 정답원핫으로 단순해집니다.
6. 자기회귀 생성: 확률분포에서 다음 글자를 샘플링하고, 그 글자를 다시 입력으로 넣는 것을 반복합니다.

## week06 RNN class에서 바뀐 부분
- forward: 출력 y를 그냥 스칼라로 내지 않고, Why의 shape을 (H,V)로 키운 뒤 softmax를 씌워 확률분포로 만듭니다.
- error: 제곱오차 대신 cross-entropy(-log(정답 확률))를 씁니다.
- backward: `dy = 2*y`(target=0 고정) 자리를 `dz = y - 정답원핫`으로 바꿉니다. 그 이후(dh, dt, dWx, dWh, dh_prev 계산)는 완전히 동일합니다.

In [1]:
import numpy as np

def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)   # 오버플로우 방지
    ex = np.exp(z)
    return ex / np.sum(ex, axis=0, keepdims=True)


class RNN:
    def __init__(self, Wx, Wh, b_h, Why, b_y):
        self.params = [Wx, Wh, b_h, Why, b_y]
        self.grads = [np.zeros_like(p) for p in self.params]
        self.grads_sum = [np.zeros_like(p) for p in self.params]
        self.cache = None

    def forward(self, x, h_prev):
        Wx, Wh, b_h, Why, b_y = self.params
        t = np.dot(Wh, h_prev) + np.dot(Wx, x) + b_h
        h_next = np.tanh(t)
        z = Why.T @ h_next + b_y
        y = softmax(z)
        self.cache = (x, h_prev, h_next, y)
        return h_next, y

    def error(self, y, target_idx):
        return -np.log(y[target_idx, 0] + 1e-12)

    def backward(self, target_idx, dh_next):
        Wx, Wh, b_h, Why, b_y = self.params
        x, h_prev, h_next, y = self.cache

        dz = y.copy()
        dz[target_idx, 0] -= 1

        dh = Why @ dz + dh_next
        dt = dh * (1 - h_next ** 2)
        db_h = dt
        dWh = np.dot(dt, h_prev.T)
        dh_prev = np.dot(Wh.T, dt)
        dWx = np.dot(dt, x.T)
        dWhy = np.dot(h_next, dz.T)
        db_y = dz

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db_h
        self.grads[3][...] = dWhy
        self.grads[4][...] = db_y

        self.grads_sum[0][...] += dWx
        self.grads_sum[1][...] += dWh
        self.grads_sum[2][...] += db_h
        self.grads_sum[3][...] += dWhy
        self.grads_sum[4][...] += db_y

        return dh_prev, self.grads

    def update(self, lr):
        for i in range(len(self.params)):
            self.params[i] -= lr * self.grads_sum[i]
        return self.params

    def reset_grads_sum(self):
        for g in self.grads_sum:
            g[...] = 0


def onehot(idx, V):
    v = np.zeros((V, 1))
    v[idx, 0] = 1.0
    return v

## 학습 + 생성 함수
- 입력은 텍스트의 마지막 글자를 제외, 정답은 첫 글자를 제외해서 한 칸 밀린 쌍을 만듭니다.
- grads_sum은 시점별로 그냥 더하기만 하므로, 시퀀스 길이(T)로 나눠서 평균 기울기로 업데이트해야 시퀀스가 길어져도 학습률이 안정적입니다.

In [2]:
def train(text, H=16, lr=0.3, rep=200, seed=0, res_len=30, first_char=None):
    chars = sorted(set(text))
    char_to_idx = {c: i for i, c in enumerate(chars)}
    idx_to_char = {i: c for i, c in enumerate(chars)}
    V = len(chars)

    idx_seq = [char_to_idx[c] for c in text]
    xs = idx_seq[:-1]
    targets = idx_seq[1:]
    T = len(xs)  # 시점의 개수

    rng = np.random.RandomState(seed)
    scale = 1 / np.sqrt(V + H)
    Wx = rng.randn(H, V) * scale
    Wh = rng.randn(H, H) * scale
    b_h = np.zeros((H, 1))
    Why = rng.randn(H, V) * scale
    b_y = np.zeros((V, 1))

    rnn = RNN(Wx, Wh, b_h, Why, b_y)
    losses = []

    for epoch in range(1, rep + 1):
        h = np.zeros((H, 1))  # 첫 은닉상태는 0
        caches = []
        ys = []
        for x_idx in xs:
            h, y = rnn.forward(onehot(x_idx, V), h)
            ys.append(y)
            caches.append(rnn.cache)

        loss = sum(rnn.error(ys[t], targets[t]) for t in range(T)) / T
        losses.append(loss)

        rnn.reset_grads_sum()
        dh_next = np.zeros((H, 1))
        for t in reversed(range(T)):
            rnn.cache = caches[t]
            dh_next, grad = rnn.backward(targets[t], dh_next)

        for g in rnn.grads_sum:
            g /= T
        rnn.update(lr)

        if epoch % 30 == 0 or epoch == 1:
            print(f'epoch {epoch:4d}  loss={loss:.4f}')

    if first_char is None:
        first_char = text[0]

    res_rng = np.random.RandomState(seed + 1)
    h = np.zeros((H, 1))
    idx = char_to_idx[first_char]
    res = [idx]
    for _ in range(res_len - 1):
        h, y = rnn.forward(onehot(idx, V), h)
        p = y.flatten()
        idx = res_rng.choice(V, p=p)
        res.append(idx)

    res_str = ''.join(idx_to_char[i] for i in res)
    print()
    print('생성 결과:', res_str)
    return rnn, losses, res_str

## 실행: "hello world" 학습

In [3]:
rnn, losses, res_str = train("hello world", H=16, lr=0.3, rep=300, seed=0, res_len=11)

epoch    1  loss=2.1203
epoch   30  loss=0.3992
epoch   60  loss=0.0961
epoch   90  loss=0.0475
epoch  120  loss=0.0307
epoch  150  loss=0.0224
epoch  180  loss=0.0175
epoch  210  loss=0.0143
epoch  240  loss=0.0121
epoch  270  loss=0.0105
epoch  300  loss=0.0092

생성 결과: hel o world


## H의 크기를 줄였을때


In [17]:
rnn, losses, res_str = train("hello world", H=4, lr=0.3, rep=300, seed=0, res_len=11)

epoch    1  loss=2.0196
epoch   30  loss=1.2716
epoch   60  loss=0.8232
epoch   90  loss=0.5869
epoch  120  loss=0.3662
epoch  150  loss=0.2235
epoch  180  loss=0.1518
epoch  210  loss=0.1126
epoch  240  loss=0.0887
epoch  270  loss=0.0728
epoch  300  loss=0.0615

생성 결과: hel  relldo


## 결과 해석
- loss가 2.12에서 0.009까지 줄어들어서, 모델이 "h 다음엔 e, e 다음엔 l, ..."이라는 패턴을 정확히 학습했습니다.
- 그런데 확률적 샘플링(np.random.choice)으로 생성하면 가끔 이상한 글자가 섞여 나올 수 있습니다. 예를 들어 'l'이 나올 확률이 99%가 넘어도, 나머지 1% 안에서 다른 글자가 뽑히는 경우가 실제로 생깁니다.
- 이걸 argmax(확률이 가장 높은 글자를 그대로 선택)로 바꿔서 생성해보면 "hello world"가 정확히 나오는 것을 확인했습니다. 즉 모델 자체는 정확하게 학습됐고, 생성 방식(샘플링 vs argmax)에 따라 결과가 달라지는 것입니다.
- 그리고 H를 줄였을 때는 은닉 상태의 크기가 줄어드므로 지금까지의 정보를 기억해두는 공간이 줄어드는 셈입니다. 그러므로 H가 작아지면 학습의 결과가 더 안좋을 수 밖에 없습니다. 다만 H가 너무 클 경우 계산량이 많아지므로 학습할 데이터의 크기가 그렇게 크지 않다면 저장공간을 낭비하게 될 수 있습니다.

In [4]:
# argmax로 생성했을 때와 비교
h = np.zeros((16, 1))
chars = sorted(set("hello world"))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
idx = char_to_idx['h']
result = [idx]
for _ in range(10):
    h, y = rnn.forward(onehot(idx, len(chars)), h)
    idx = int(np.argmax(y))
    result.append(idx)
print('argmax로 생성:', ''.join(idx_to_char[i] for i in result))

argmax로 생성: hello world


## 번외) 같은 단어를 반복하는 예제

In [43]:
rnn, losses, res_str = train("hello hello hello hello hello", H=4, lr=0.3, rep=300, seed=0, res_len=29)

epoch    1  loss=1.6679
epoch   30  loss=0.6276
epoch   60  loss=0.2059
epoch   90  loss=0.1168
epoch  120  loss=0.0802
epoch  150  loss=0.0606
epoch  180  loss=0.0485
epoch  210  loss=0.0402
epoch  240  loss=0.0343
epoch  270  loss=0.0299
epoch  300  loss=0.0264

생성 결과: hel o hello hello hello hello


- 아까처럼 H를 4로 줄였지만 이번에는 같은 단어를 반복하는 문자열을 학습시켰습니다.
- 그 결과 첫 예제와 달리 비교적 결과가 잘 나온 것을 확인할 수 있습니다.
- hello world는 글자 종류가 8개이고 hello는 6개(공백 포함) 이므로 H의 크기가 작아도 은닉 상태가 감당해야할 부담이 적습니다.
- 패턴이 단순하고 반복적인 문자열일수록 학습이 더 빨리 되는것을 알 수 있고 데이터에 반복성이 있다면 은닉 상태의 크기가 작아도 학습에 지장이 없습니다. 

## 결론
- RNN의 은닉상태 계산 부분(week06 그대로)에 softmax 출력층과 cross-entropy 손실을 얹으면, 다음 글자를 예측하는 언어모델을 만들 수 있습니다.
- 학습된 언어모델에서 확률분포를 뽑아 자기회귀적으로 반복하면 문장(짧은 문자열)을 생성할 수 있습니다.
- 생성 결과가 항상 정확하지 않을 수 있는데, 이는 모델이 잘못 학습된 것이 아니라 확률적 샘플링 과정에서 낮은 확률의 글자가 뽑히는 경우가 있기 때문입니다.